# Editor gallery — how editing fails, by editor family

**Thread:** `editability/` · **Canonical names/metrics:** `../METRICS_AND_EDITORS.md` · **Conventions:**
`../../../../CLAUDE.md`. **Model:** GRU `runs/controls/H256` — H=256, trained on `datasets/4_fixed_refl_inview`
(the split everything here is evaluated on) and the **base checkpoint every fine-tuned arm was trained from**.
**Data:** `edits` split, N=64 held-out edits, `ef = 20`, K=15 rollout steps. **No retraining** — the five
learned arms are loaded from `runs/trained_editability/`.

## What this notebook is for

It produces the **three slide figures** that replace the old single all-editors waterfall in
`../00_master_editability.ipynb` (Fig 5a). One figure per editor family, so the *shape* of each family's
failure is visible rather than averaged into a bar chart:

| figure | family | the question |
|---|---|---|
| **Fig A** | **Standard editors** — training-free writes to `h` | do any of the structural write mechanisms move the object? |
| **Fig B** | **Learned editors** — the world model or the editor was trained | does *training* for editability fix it? |
| **Fig C** | **Oracle editors** — given ground-truth access | what does a *working* edit look like, and which oracles actually work? |

> **Why this notebook and not the master.** `CLAUDE.md` requires the master synthesis notebook to stay
> **lightweight** — "recompute only the cheap things; cite the rest with source-notebook provenance". Seventeen
> editors including a 120-iteration local-PCA geodesic and five fine-tuned checkpoints is not that. The heavy
> computation lives here; the master displays the resulting figures and cites this notebook.

> **Provenance note — the model changed.** The master notebook's §1–§4 use
> `runs/gru/3_dset3_gru_persistentids_inview_400epochs`, which was trained on **dataset 3** and evaluated on
> dataset 4. This gallery uses **`runs/controls/H256`** instead, for two reasons: it is trained on the dataset it
> is evaluated on, and **every fine-tuned arm in Fig B is fine-tuned from it** (`base: runs/controls/H256` in
> each run's `config.json`), so Fig B is only interpretable against that base. Numbers here therefore will not
> match the master's older §4 tables exactly.

## Reading the waterfalls

Canonical spec (`CLAUDE.md`): **gray colormap on a dark background**; the **6 rows above the dashed orange line**
are the actual *noisy* observations the model was teacher-forced on; **every row below that line is that row's
own free-run**, starting at step 0, which decodes sim frame `ef`. **Green solid** = where the object should be
after the edit. **Red dashed** = where it was before (the "ghost" it must vacate). A successful edit moves the
bright band from the red line to the green line and leaves nothing behind.

**One exception, and it is labelled in the figure:** *First Obs. TF* is fed `obs[ef]`, so it **leads every other
row by one frame**. Per `CLAUDE.md` it is labelled rather than re-aligned.

Each row carries its **Edit Index** and **fidelity ratio**. Edit Index: **+1** = the output *is* the world where
the edit happened, **−1** = the world where it did not, **0** = equidistant *or garbage*. Fidelity ratio **> 1
means the edit left the rollout further from the truth than doing nothing** — no success claim survives it.

## The editors — canonical names (this list is now the source of truth)

These names are the repo standard from 2026-08-05 and are mirrored into `../METRICS_AND_EDITORS.md`.
**PI** = pseudoinverse. **TF** = teacher forcing.

### Standard editors — training-free writes to `h`

| name | mechanism |
|---|---|
| **Pseudoinverse Injection** | `Δ = A⁺(target − (Ah+b))`; the minimum-norm write that sets the linear position readout to the target |
| **Global PCA Projection (PI)** | alternating projections (POCS): inject ↔ project onto the global 90%-variance PCA subspace, 50 rounds |
| **Local PCA Geodesic @120 (PI)** | constant-step walk toward the injection target, re-projecting onto a **freshly refit local**-PCA tangent (64 nearest neighbours) at each of 120 steps |
| **MLP Grad Steering** | Adam on `h` through a **frozen MLP (pos,vel) probe** (d=8) toward the target, 200 steps |
| **Multistep Steering (PI) @16** | 16 rounds of: nudge the readout a fraction η=0.2 toward the target, **decode the model's own prediction and feed it back** (observe-and-settle). Both objects' targets are re-asserted every round. The fed-back observation is **model-generated, never rendered** — that is what separates it from the freeze-time oracle. |
| **Multistep Steering w/ PCA (PI) @16** | as above, plus a projection onto the global PCA subspace each round |
| **Iterative Nullspace Projection @29 (R² corrected)** | fit a position probe, delete its 4-dim row space from `h`, refit, ×29 → 29 mutually orthogonal probes spanning 116 dims. Inject into **all** of them at once (exactly solvable; the blocks do not interfere). Targets are **shrunk by each probe's own R²**, `target_k = μ + R²_k(target − μ)`, because a probe with R² ≈ 0 reads the population mean on a genuine edited state, not the target. |

### Learned editors — the model or the editor was trained

All five write through **Pseudoinverse Injection using their own frozen probe** (saved per run as
`frozen_probe.npz`), except the last, which uses its learned editor network. Each is a **different world model**,
so each row carries **its own unsteered baseline** — a fine-tune that damages prediction raises the unsteered
Edit Index for free.

| name | what was trained |
|---|---|
| **Finetuned Model · light · 300 steps** | the world model, 300 steps, retention weight 1.0 |
| **Finetuned Model · heavy · 3000 steps** | the world model, 3000 steps, retention weight 1.0 |
| **Finetuned Model · heavy, no retention · 3000 steps** | as above with retention weight **0** — the ablation that shows what retention is holding together |
| **Finetuned Model · heavy, object-0 edits only · 3000 steps** | as above but trained only on object-0 edits; the content-generalisation control |
| **Trained Editor · 3000 steps** | the **world model is frozen**; a network `E_θ(h, target) → Δh` (2×512 MLP) is trained instead |

### Oracle editors — given ground-truth access

| name | mechanism |
|---|---|
| **Freeze-time Interp. TF @8** | freeze the world and teacher-force **8 externally rendered** frames in which the object interpolates to the target, then unfreeze |
| **Counterfactual Overwriting** | render a whole fabricated history in which the object always travelled toward the target, teacher-force it, and overwrite the pre-edit state |
| **First Obs. TF** | teacher-force **one** frame: the real (noisy) post-edit observation `edits.obs[ef]`. The model simply gets to *see* the teleport. **Leads the other rows by one frame.** |
| **Decoder Grad Steering k=1** | Adam on `h` so the decoder renders the **ground-truth edit-frame observation** exactly |
| **Decoder Grad Steering k=15** | Adam on `h` so the **whole 15-step rollout** matches the ground-truth post-edit sequence (backpropagating through the dynamics) |

In [ ]:
# [1] Setup: load the standard H256 GRU, the edit set, probes, subspaces, and the canonical §4 zones.
import os, sys, time, json
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
from dataclasses import replace
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown
from tqdm.auto import tqdm

import pim.eval as ev
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition
from pim.editors import (probe_decomposition, inject_state, fit_state_subspace,
                         project_to_subspace, fit_local_subspace, manifold_steer, gradient_steer)
from pim.world_models import load_checkpoint, load_dataset, make_test_loader
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT = "../../../.."
N_OBJ, N_EDIT, K_ROLL, N_CTX, N_FT = 2, 64, 15, 6, 8
OUT = "figures"; os.makedirs(OUT, exist_ok=True)

MODEL, INFO = load_checkpoint(f"{ROOT}/runs/controls/H256/best_model.pt", device=DEVICE)
Hd = MODEL.hidden_size
bundle = load_dataset(f"{ROOT}/datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]; R = edits.obs_res; DT = float(sim["dt"])
N = min(N_EDIT, edits.n_samples)
with h5py.File(edits.h5_path, "r") as f:
    VEL_ALL = f["velocities"][:, :, :N_OBJ, :].astype(np.float32)

# teacher-forced bank for probes / PCA subspace / local bank
loader = make_test_loader(test, batch_size=512, num_workers=4)
_, STATES = ev.teacher_force(MODEL, loader, device=DEVICE)          # (Ntest, 39, H)
pos_tf = test.positions[:, :-1, :N_OBJ, :]
vis_tf = test.is_visible[:, :-1, :N_OBJ].all(axis=2)
with h5py.File(test.h5_path, "r") as f:
    vel_tf = f["velocities"][:, :-1, :N_OBJ, :].astype(np.float32)
posvel_tf = np.concatenate([pos_tf.reshape(*pos_tf.shape[:2], -1),
                            vel_tf.reshape(*vel_tf.shape[:2], -1)], -1)

sdef = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
lin = LinearExtractor(Hd, sdef, use_lstsq=True); lin.fit(STATES, pos_tf, mask=vis_tf, device=DEVICE)
lin = lin.to(DEVICE).eval()
A, b_, A_pinv = probe_decomposition(lin)

sdef_pv = StateDefinition(name="posvel", state_shape=(N_OBJ * 4,), extract_fn=lambda x: x)
MLP_PV = MLPExtractor(Hd, sdef_pv, mlp_hidden=128, n_epochs=30, lr=5e-3)
MLP_PV.fit(STATES, posvel_tf, mask=vis_tf, device=DEVICE); MLP_PV = MLP_PV.to(DEVICE).eval()

SUB = fit_state_subspace(STATES, var_threshold=0.90)
SUB = replace(SUB, mean=SUB.mean.to(DEVICE), basis=SUB.basis.to(DEVICE),
              explained_variance_ratio=SUB.explained_variance_ratio.to(DEVICE))
bank_all = STATES.reshape(-1, Hd)
bidx = np.random.RandomState(0).choice(bank_all.shape[0], size=min(50_000, bank_all.shape[0]), replace=False)
BANK = torch.from_numpy(bank_all[bidx]).float().to(DEVICE)

# ── the edit set ──────────────────────────────────────────────────────────────
oe  = edits.edit_object[:N].astype(int)
pos = edits.positions[:N][:, :, :N_OBJ, :].astype(np.float32)
TGT, PRE = pos[:, ef].copy(), pos[:, ef - 1]
tgt   = torch.from_numpy(TGT.reshape(N, N_OBJ * 2)).float().to(DEVICE)
tgt_pv = torch.from_numpy(np.concatenate(
    [TGT.reshape(N, N_OBJ * 2), VEL_ALL[:N, ef].reshape(N, N_OBJ * 2)], 1)).float().to(DEVICE)

@torch.no_grad()
def warm(obs_np, upto, model=MODEL):
    o = torch.from_numpy(obs_np).float().to(DEVICE); st = None
    for t in range(upto):
        _, st = model.step(o[:, t], st)
    return model.flat_state(st)

@torch.no_grad()
def roll(h, steps=K_ROLL, model=MODEL):
    st = model.state_from_flat(torch.as_tensor(h, dtype=torch.float32, device=DEVICE))
    out = [model.decode(st)]
    for _ in range(steps - 1):
        p, st = model.predict_step(st); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

H0 = warm(edits.obs[:N].astype(np.float32), ef)

gt_roll = edits.clean_obs[:N, ef:ef + K_ROLL, :].astype(np.float32)
CTX     = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)      # the NOISY frames actually teacher-forced
ZONES = build_edit_zones(pre_pos=PRE, tgt_pos=TGT, pre_vel=VEL_ALL[:N, ef - 1, :N_OBJ, :],
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N, ef:ef + K_ROLL, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
print(f"model {INFO.run_name} H={Hd} | N={N} edits | ef={ef} | K={K_ROLL}")
print(f"ray zones/sample: target {ZONES.target.sum(1).mean():.1f}, ghost {ZONES.ghost.sum(1).mean():.1f}, "
      f"differing {ZONES.differing.sum(1).mean():.1f}")

In [ ]:
# [2] Renderers for the oracle editors, and the scoring helper used by every family.
REFL = np.array([sim["refl_min"], sim["refl_max"]], np.float32)
RAD  = np.array([sim["radius"]] * N_OBJ, np.float32)
COL  = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
OBS_NOISE = float(sim["obs_noise_std"])
def _cfg(nf, noise):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                     n_objects=N_OBJ, radius=sim["radius"], n_frames=nf, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                     obs_noise_std=noise, boundary="open", always_in_frustum=False)
def render_traj(pos_seq, noise=0.0):
    _, _, inten = render_scene(Scene(positions=pos_seq, velocities=np.zeros_like(pos_seq), radii=RAD,
                                     colors=COL, reflectivities=REFL, config=_cfg(len(pos_seq), noise)))
    return inten.astype(np.float32)

cf_obs = np.zeros((N, ef, R), np.float32)         # counterfactual history
ft_obs = np.zeros((N, N_FT, R), np.float32)       # freeze-time interpolation frames
t_idx = np.arange(ef)
for i in range(N):
    o, other = oe[i], 1 - oe[i]
    v = VEL_ALL[i, ef, o]
    cf = np.zeros((ef, N_OBJ, 2), np.float32)
    cf[:, o]     = TGT[i, o][None, :] - v[None, :] * (ef - t_idx)[:, None] * DT
    cf[:, other] = pos[i, :ef, other]
    cf_obs[i] = render_traj(cf)
    fr = np.zeros((N_FT, N_OBJ, 2), np.float32)
    for j in range(N_FT):
        fr[j, o] = PRE[i, o] + ((j + 1) / N_FT) * (TGT[i, o] - PRE[i, o]); fr[j, other] = TGT[i, other]
    ft_obs[i] = render_traj(fr, OBS_NOISE)

@torch.no_grad()
def continue_from(h, frames, model=MODEL):
    st = model.state_from_flat(h); o = torch.from_numpy(frames).float().to(DEVICE)
    for t in range(frames.shape[1]):
        _, st = model.step(o[:, t], st)
    return model.flat_state(st)

CARDS, ROLLS, UNST = {}, {}, {}
def register(name, h, model=MODEL, unsteered_name="Unsteered"):
    """Roll a state out, score it on the canonical §4 set, and store it for the figures."""
    rl = roll(h, model=model); ROLLS[name] = rl
    c = edit_scorecard(rl, ZONES, gt_roll)
    base = CARDS[UNST.get(name, unsteered_name)] if name != unsteered_name else c
    c["fidelity_ratio"] = 1.0 if name == unsteered_name else fidelity_ratio(c, base)
    CARDS[name] = c
    return c

register("Unsteered", H0)
print(f"Unsteered Edit Index {CARDS['Unsteered']['edit_index']:+.2f} — this model's own '−1' end")

In [ ]:
# [3] STANDARD EDITORS (7) — training-free writes to h.
t0 = time.perf_counter()
STD = []

# 1 — Pseudoinverse Injection
h_pi = inject_state(H0, tgt, A, A_pinv, b_)
STD.append("Pseudoinverse Injection"); register("Pseudoinverse Injection", h_pi)

# 2 — Global PCA Projection (PI): alternating inject <-> project (POCS)
STD.append("Global PCA Projection (PI)")
register("Global PCA Projection (PI)",
         manifold_steer(H0, tgt, lambda h, t: inject_state(h, t, A, A_pinv, b_), SUB, n_iters=50))

# 3 — Local PCA Geodesic @120 (PI): constant-step walk, refit LOCAL tangent every step
@torch.no_grad()
def local_pca_geodesic(h_start, target, k_iters=120, k_local=64):
    const = 0.34 * float((inject_state(h_start, target, A, A_pinv, b_) - h_start).norm(dim=-1).mean())
    out = torch.empty_like(h_start)
    for i in tqdm(range(h_start.shape[0]), desc="Local PCA Geodesic", leave=False):
        h, t = h_start[i:i + 1], target[i:i + 1]
        for _ in range(k_iters):
            d = inject_state(h, t, A, A_pinv, b_) - h
            nrm = d.norm(); h = h + const * (d / nrm if float(nrm) > 1e-12 else d)
            h = project_to_subspace(h, fit_local_subspace(BANK, h[0], k_neighbors=k_local,
                                                          var_threshold=0.90, bank_size=50_000))
        out[i] = h[0]
    return out
STD.append("Local PCA Geodesic @120 (PI)")
register("Local PCA Geodesic @120 (PI)", local_pca_geodesic(H0, tgt))

# 4 — MLP Grad Steering: Adam on h through the frozen MLP (pos,vel) probe
outs = []
for i in tqdm(range(N), desc="MLP Grad Steering", leave=False):
    hi, _ = gradient_steer(H0[i:i + 1], tgt_pv[i:i + 1], MLP_PV, n_steps=200, lr=0.01)
    outs.append(hi)
STD.append("MLP Grad Steering"); register("MLP Grad Steering", torch.cat(outs, 0))

# 5/6 — Multistep Steering: nudge the readout, then feed the model its OWN decoded obs back.
#       The observation is MODEL-GENERATED, never rendered — that is what separates it from freeze-time.
@torch.no_grad()
def multistep_steer(h_init, target, S=16, eta=0.2, project=False):
    h = h_init.clone()
    for _ in range(S):
        r = h @ A.T + b_
        h = inject_state(h, r + eta * (target - r), A, A_pinv, b_)   # partial move, both objects re-asserted
        if project:
            h = project_to_subspace(h, SUB)
        st = MODEL.state_from_flat(h)
        _, st = MODEL.step(MODEL.decode(st), st)                     # observe-and-settle on its OWN prediction
        h = MODEL.flat_state(st)
    return h
STD.append("Multistep Steering (PI) @16"); register("Multistep Steering (PI) @16", multistep_steer(H0, tgt))
STD.append("Multistep Steering w/ PCA (PI) @16")
register("Multistep Steering w/ PCA (PI) @16", multistep_steer(H0, tgt, project=True))

# 7 — Iterative Nullspace Projection @29 (R² corrected)
Hb = STATES.reshape(-1, Hd).astype(np.float64)
Yb = pos_tf.reshape(-1, N_OBJ * 2).astype(np.float64)
ntr = int(0.8 * len(Hb))
def _fit(X, Y):
    Aug = np.concatenate([X[:ntr], np.ones((ntr, 1))], 1)
    sol, *_ = np.linalg.lstsq(Aug, Y[:ntr], rcond=None)
    p = X[ntr:] @ sol[:-1] + sol[-1]
    r2 = float(1 - ((p - Y[ntr:]) ** 2).sum() / ((Y[ntr:] - Y[:ntr].mean(0)) ** 2).sum())
    return sol[:-1].T, sol[-1], r2
Xr = Hb.copy(); CASCADE = []
for _ in range(29):
    Ak, bk, r2k = _fit(Xr, Yb)
    _, s, vt = np.linalg.svd(Ak, full_matrices=False)
    rk = int((s > s[0] * 1e-8).sum()); Bk = vt[:rk].T
    CASCADE.append(dict(A=Ak, b=bk, r2=r2k, pinv=np.linalg.pinv(Ak)))
    Xr = Xr - (Xr @ Bk) @ Bk.T
MU_POS = Yb[:ntr].mean(0)
h0_np = H0.cpu().numpy().astype(np.float64); tgt_np = TGT.reshape(N, N_OBJ * 2).astype(np.float64)
dh_inlp = np.zeros_like(h0_np)
for c_ in CASCADE:                       # R²-shrunk target: what a REAL edited state would read through probe k
    t_k = MU_POS + c_["r2"] * (tgt_np - MU_POS)
    dh_inlp += (t_k - (h0_np @ c_["A"].T + c_["b"])) @ c_["pinv"].T
STD.append("Iterative Nullspace Projection @29 (R² corrected)")
register("Iterative Nullspace Projection @29 (R² corrected)",
         torch.from_numpy((h0_np + dh_inlp).astype(np.float32)).to(DEVICE))
print(f"\nstandard editors built in {time.perf_counter()-t0:.0f}s | cascade ranks "
      f"{sorted(set(int(np.linalg.matrix_rank(c['A'])) for c in CASCADE))}, "
      f"R² {CASCADE[0]['r2']:.3f} -> {CASCADE[-1]['r2']:.3f}")

In [ ]:
# [4] LEARNED EDITORS (5) — each is a DIFFERENT world model, so each gets its own unsteered baseline.
import torch.nn as nn
class AmortizedEditor(nn.Module):
    """E_θ(h, target) → Δh, exactly as trained by scripts/train_editable_gru.py."""
    def __init__(self, hidden, target_dim=N_OBJ * 2, width=512):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hidden + target_dim, width), nn.ReLU(),
                                 nn.Linear(width, width), nn.ReLU(), nn.Linear(width, hidden))
    def forward(self, h, target):
        return h + self.net(torch.cat([h, target], -1))

LEARNED_RUNS = [
    ("Finetuned Model · light · 300 steps",                       "FT_light"),
    ("Finetuned Model · heavy · 3000 steps",                      "FT_heavy"),
    ("Finetuned Model · heavy, no retention · 3000 steps",        "FT_heavy_noret"),
    ("Finetuned Model · heavy, object-0 edits only · 3000 steps", "FT_heavy_obj0"),
    ("Trained Editor · 3000 steps",                               "AMORT"),
]
LRN = []
for name, code_ in LEARNED_RUNS:
    d = f"{ROOT}/runs/trained_editability/{code_}"
    m, _ = load_checkpoint(f"{d}/best_model.pt", device=DEVICE)
    z = np.load(f"{d}/frozen_probe.npz")
    Wf = torch.tensor(z["W"], device=DEVICE); bf = torch.tensor(z["b"], device=DEVICE)
    Wp = torch.tensor(z["W_pinv"], device=DEVICE)
    h0_m = warm(edits.obs[:N].astype(np.float32), ef, model=m)      # this model's OWN pre-edit state
    un = f"Unsteered · {code_}"
    register(un, h0_m, model=m, unsteered_name=un)
    if code_ == "AMORT":
        ed = AmortizedEditor(m.hidden_size).to(DEVICE)
        ed.load_state_dict(torch.load(f"{d}/amortized_editor.pt", map_location=DEVICE)["editor_state"])
        ed.eval()
        with torch.no_grad():
            h_ed = ed(h0_m, tgt)
    else:
        h_ed = h0_m + (tgt - (h0_m @ Wf + bf)) @ Wp                 # PI injection through its OWN frozen probe
    UNST[name] = un
    LRN.append(name); register(name, h_ed, model=m)
    print(f"{name:<52s} unsteered {CARDS[un]['edit_index']:+.2f} -> edited "
          f"{CARDS[name]['edit_index']:+.2f} (fidelity {CARDS[name]['fidelity_ratio']:.2f})")

In [ ]:
# [5] ORACLE EDITORS (5) — given ground-truth access.
t0 = time.perf_counter()
ORC = []

ORC.append("Freeze-time Interp. TF @8"); register("Freeze-time Interp. TF @8", continue_from(H0, ft_obs))
ORC.append("Counterfactual Overwriting"); register("Counterfactual Overwriting", warm(cf_obs, ef))

# First Obs. TF — teacher-force ONE frame, the real (noisy) post-edit observation. LEADS BY ONE FRAME.
obs_ef = edits.obs[:N, ef:ef + 1, :].astype(np.float32)
ORC.append("First Obs. TF"); register("First Obs. TF", continue_from(H0, obs_ef))

gt_ef = torch.from_numpy(edits.clean_obs[:N, ef, :]).float().to(DEVICE)
gt_seq = torch.from_numpy(gt_roll).float().to(DEVICE)

def decoder_grad(h_init, k, n_iter=400, lr=0.05):
    """k=1: match the GT edit-frame observation through the decoder.
       k=15: match the WHOLE GT rollout, backpropagating through the dynamics."""
    h = h_init.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            st = MODEL.state_from_flat(h)
            if k == 1:
                loss = ((MODEL.decode(st) - gt_ef) ** 2).mean()
            else:
                outs = [MODEL.decode(st)]
                for _s in range(k - 1):
                    p, st = MODEL.predict_step(st); outs.append(p)
                loss = ((torch.stack(outs, 1) - gt_seq) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach()

ORC.append("Decoder Grad Steering k=1");  register("Decoder Grad Steering k=1",  decoder_grad(H0, 1))
ORC.append("Decoder Grad Steering k=15"); register("Decoder Grad Steering k=15", decoder_grad(H0, K_ROLL))
print(f"oracles built in {time.perf_counter()-t0:.0f}s")

rows = ["| family | editor | Edit Index (step 0) ↑ | at step 14 | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ↓ |",
        "|---|---|---|---|---|---|---|---|---|"]
for fam, names in [("reference", ["Unsteered"]), ("standard", STD), ("learned", LRN), ("oracle", ORC)]:
    for n_ in names:
        c = CARDS[n_]
        rows.append(f"| {fam} | {n_} | **{c['edit_index']:+.2f}** | {c['edit_index_by_step'][-1]:+.2f} | "
                    f"{c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | "
                    f"{c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown(f"**Table 1 — every editor on the canonical §4 set.** N={N} held-out edits, GRU H256. Learned "
                 "arms are scored against **their own** unsteered row (printed in cell [4]); everything else "
                 "against the shared `Unsteered` row. **fidelity > 1 = worse than doing nothing.**\n\n"
                 + "\n".join(rows)))

In [ ]:
# [6] The one waterfall helper every figure below goes through (canonical CLAUDE.md spec).
import textwrap
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"

def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
TGT_CX = np.array([_cx(ZONES.target[i]) for i in range(N)])
GHO_CX = np.array([_cx(ZONES.ghost[i]) for i in range(N)])

teleport = np.linalg.norm(TGT[np.arange(N), oe] - PRE[np.arange(N), oe], axis=-1)
SAMPLES = [int(i) for i in np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:4]]
print("samples (largest teleports with a well-defined ghost region):", SAMPLES)

def waterfall(cols, title, fname, extra_legend=(), dpi=170):
    """cols: list of (editor name, body (N,K,R)). ONE EDITOR PER COLUMN, samples down the rows.

    Canonical spec: gray on dark; N_CTX NOISY context frames above a dashed edit line; below it every
    panel is that column's OWN free-run from step 0 (which decodes sim frame `ef`). No shared
    teacher-forced row — only an oracle-observation editor ever sees that frame. Sized 16:9 for slides.
    Column headings are the editor names alone; all metrics live in Table 1, not on the figure."""
    nr, nc = len(SAMPLES), len(cols)
    fig, axes = plt.subplots(nr, nc, figsize=(19.2, 10.8), squeeze=False, facecolor=DARK)
    wrap = max(13, int(155 / nc))
    fs = 14.0 if nc <= 7 else 12.0
    for r, smp in enumerate(SAMPLES):
        for c, (lab, body) in enumerate(cols):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([CTX[smp], body[smp]], 0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                      interpolation="nearest")
            for sp in ax.spines.values():
                sp.set_edgecolor(TICK); sp.set_linewidth(0.8)
            ax.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.3, ls="--", alpha=0.95)
            if not np.isnan(TGT_CX[smp]): ax.axvline(TGT_CX[smp], color=TARGET_C, lw=1.6, alpha=0.95)
            if not np.isnan(GHO_CX[smp]): ax.axvline(GHO_CX[smp], color=GHOST_C, ls="--", lw=1.6, alpha=0.95)
            if r == 0:
                ax.set_title("\n".join(textwrap.wrap(lab, wrap)), fontsize=fs, color=TXT, pad=9)
            ax.set_xticks([]); ax.set_yticks([])
            if c == 0:
                ax.set_ylabel(f"sample {smp}", fontsize=12.5, color=TXT, rotation=0,
                              ha="right", va="center", labelpad=14)
    handles = [Line2D([0],[0], color=TARGET_C, lw=3.0, label="target — where the object should end up"),
               Line2D([0],[0], color=GHOST_C, ls="--", lw=3.0, label="ghost — where it was before the edit"),
               Line2D([0],[0], color=EDIT_C, ls="--", lw=3.0,
                      label=f"edit applied here · {N_CTX} noisy context frames above · every panel below "
                            f"is that column's own free-run from sim frame {ef}")]
    for txt in extra_legend:
        handles.append(Line2D([0],[0], color=DARK, lw=0, label=txt))
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=11.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(title, y=0.995, fontsize=21, color=TXT)
    # left margin must clear the row labels — at 0.052 they were clipped to "ample 26"
    fig.subplots_adjust(top=0.775, bottom=0.015, left=0.078, right=0.995, hspace=0.05, wspace=0.035)
    fig.savefig(f"{OUT}/{fname}", dpi=dpi, facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)

def colset(names):
    """GT and the shared unsteered rollout, then one column per editor."""
    return [("GT (sim)", gt_roll), ("Unsteered", ROLLS["Unsteered"])] + [(n, ROLLS[n]) for n in names]

In [ ]:
# [7] Fig A — STANDARD editors: training-free writes to h.
waterfall(colset(STD), "Standard Editors", "figA_standard_editors.png")

In [ ]:
# [8] Fig B — LEARNED editors. Each column is a DIFFERENT world model; the single `Unsteered` column is the
# BASE model (runs/controls/H256), shown as the shared reference rather than one no-edit column per arm.
# Per-arm unsteered indices are printed in cell [4] and tabulated in Table 1 — read the numbers there, since
# an index is only comparable to its own model's baseline.
waterfall(colset(LRN), "Learned Editors", "figB_learned_editors.png")

In [ ]:
# [9] Fig C — ORACLE editors: what a working edit looks like, and which oracles actually work.
# `First Obs. TF` is fed obs[ef], so it leads the other columns by one frame — noted in the legend
# (CLAUDE.md requires it be labelled, never re-aligned to the others).
waterfall(colset(ORC), "Oracle Editors", "figC_oracle_editors.png",
          extra_legend=("note: First Obs. TF is fed the post-edit observation, so it leads by one frame",))

### Current results (updated 2026-08-05)

*GRU `runs/controls/H256`, N=64 held-out edits. Full numbers in Table 1; this is what the three figures show.*

**Standard editors (Fig A) — all seven fail, and two of them fail by *degrading* rather than by doing nothing.**
The object stays on the red ghost locator in every row. Best is **Iterative Nullspace Projection @29 (R²
corrected)** at **−0.37** (fidelity 0.93) against an unsteered floor of **−0.68**; **Pseudoinverse Injection**
is visually indistinguishable from unsteered at **−0.66**. The two multistep rows are the cautionary ones:
**Multistep Steering (PI) @16** posts the best-looking index of the family (**−0.22**) but with **fidelity
1.32** and collateral RMSE **0.429** against unsteered's 0.127 — it drags *both* objects. Its index moved
because the output got worse, which is exactly what the Edit Index is built to read as ≈0 rather than reward.

**Learned editors (Fig B) — training helps, and none of it reaches an edit.** Every arm must be read against
its own unsteered row, which is why each is paired with its own no-edit rollout in the figure. Best is the
**Trained Editor · 3000 steps** (world model frozen): **−0.68 → −0.14**, fidelity 0.68 — a gain of +0.54, the
largest of any mechanism here, and still short of the 0 mark. Fine-tuning the world model buys less
(**−0.61 → −0.47** heavy). **No retention** is the instructive failure: its unsteered index *rises* to −0.39
purely because its prediction degraded, so its apparently-better −0.30 is mostly scale movement, not editing.

**Oracle editors (Fig C) — the k axis is the headline.** **Decoder Grad Steering k=1** scores **+0.97 at the
edit frame** — it renders the target essentially perfectly — and then **collapses to +0.08 by step 14**, which
the figure shows as vertical striping: it lands off-manifold and the dynamics reject it. Optimising the *whole*
rollout instead fixes exactly that: **Decoder Grad Steering k=15** holds **+0.83 → +0.77** at fidelity **0.20**,
the only editor here that both lands and persists. **Counterfactual Overwriting** remains the cleanest
mechanism-realistic oracle (**+0.70 → +0.45**), **Freeze-time Interp. TF @8** works but decays
(**+0.52 → +0.26**), and **First Obs. TF** — one frame of real evidence — only reaches **−0.08**, better than
every standard editor yet still on the unedited side.

**The through-line the three figures make visible.** Nothing that writes to `h` from a *readout* works. The two
mechanisms that do work either feed the model **externally rendered observations** (freeze-time,
counterfactual) or optimise `h` against the **full future** rather than one frame (k=15). Every one-shot
readout-derived write either leaves the object where it was or degrades the rollout.

**Caveats.** One model, one seed, N=64. Fine-tuned arms are five different world models. The
`Local PCA Geodesic` uses k=120; a k=600 budget extension is in the master notebook's §4 and does not change
the conclusion. `Multistep Steering` is re-implemented here on H256, so its numbers differ from
`../multistep_steering.ipynb`, which used a different checkpoint.